# Демо: публикуем модель на HuggingFace Hub

Прокликай по ячейкам сверху вниз — покажем, как опубликовать **готовую** модель.
Стартуем «с этого места»: модель уже обучена и лежит на Hub —
[HOhus/pushkin-nano-bpe](https://huggingface.co/HOhus/pushkin-nano-bpe) (из модуля 5).
Здесь только публикация; что такое форматы и локальный запуск (GGUF, MLX, LM Studio) —
в [уроке 5.1](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-1-publish-model/).

In [ ]:
# Окружение + логин. В Colab pip уже почти всё ставит.
!pip install -q transformers huggingface_hub
import os
from huggingface_hub import login, whoami
# Залогинься: либо положи токен (scope WRITE) в переменную HF_TOKEN,
# либо запусти отдельно:  from huggingface_hub import notebook_login; notebook_login()
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
HF_USER = whoami()["name"]
print("залогинен как:", HF_USER)

## Шаг 0. Берём готовую модель

Загрузим обученную модель с Hub в две строки — это и есть «нативный» артефакт.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
SRC = "HOhus/pushkin-nano-bpe"          # уже обученная модель из модуля 5
model = AutoModelForCausalLM.from_pretrained(SRC)
tok   = AutoTokenizer.from_pretrained(SRC)
print("взяли модель:", model.config.model_type, "| параметров:", model.num_parameters())

## Шаг 1. Из каких файлов состоит модель

`save_pretrained` кладёт веса (`model.safetensors`), конфиг (`config.json`) и токенизатор в папку.

In [ ]:
# Из каких файлов состоит модель — сохраним в папку и посмотрим.
model.save_pretrained("my-model")
tok.save_pretrained("my-model")
print("файлы модели:")
for f in sorted(os.listdir("my-model")):
    print("  ", f, "-", os.path.getsize(os.path.join("my-model", f)), "bytes")

## Шаг 2. Публикуем под своим именем

`push_to_hub` создаёт репозиторий и заливает файлы. Нужен токен со scope `write`.

In [ ]:
REPO = f"{HF_USER}/pushkin-nano-bpe"   # станет huggingface.co/<твой-username>/pushkin-nano-bpe
model.push_to_hub(REPO)
tok.push_to_hub(REPO)
print("опубликовано:", "https://huggingface.co/" + REPO)

## Шаг 3. Проверяем

Открой ссылку выше: там карточка, файлы (`model.safetensors`, `config.json`, токенизатор) и кнопка **Use this model**. И заберём модель обратно — докажем, что публикация рабочая:

In [ ]:
back = AutoModelForCausalLM.from_pretrained(REPO)
print("скачали обратно с Hub, параметров:", back.num_parameters())

---

**Итог:** собрать файлы → `push_to_hub` → модель на Hub, её заберёт любой одной командой.
Дальше в [уроке 5.1](https://itrubnikov.github.io/Train_of_Thought/docs/modules/05-1-publish-model/) —
форматы (GGUF/MLX) и локальный запуск в LM Studio.